## Buffer Zone Creation

####  *AUTHOR:* Ehsan Farahbakhsh
####  *CONTACT:* e.farahbakhsh@sydney.edu.au

In [ ]:
from ipywidgets import interact

import gplately
from gplately import PlateModelManager

from lib.main import *
from lib.plot import *

from parameters import parameters

In [ ]:
# Timespan for analysis
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]
time_max = parameters["timespan"]["max"]
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

plate_model_name = parameters["plate_model_name"]
anchor_plate_id = parameters["anchor_plate_id"]
buffer_distance = parameters["buffer_distance"]

plate_model_dir = parameters["plate_model_dir"]
outputs_dir = parameters["outputs_dir"]
buffer_zones_dir = parameters["buffer_zones_dir"]
buffer_zone_maps_dir = parameters["buffer_zone_maps_dir"]

buffer_zones_dir = os.path.join(outputs_dir, buffer_zones_dir)
buffer_zone_maps_dir = os.path.join(outputs_dir, buffer_zone_maps_dir)

n_jobs = 20

In [ ]:
# heidari26
rotation_model = plate_model_dir+"/CombinedRotations.rot"

topology_features = [
    plate_model_dir+"/Deforming_Networks_Active.gpml",
    plate_model_dir+"/Deforming_Networks_Inactive.gpml",
    plate_model_dir+"/Feature_Geometries.gpml",
    plate_model_dir+"/Iran-Afghan_Topological Points.gpml",
    plate_model_dir+"/Iran-Afghan-Turan_Topological Line.gpml",
    plate_model_dir+"/Plate_Boundaries.gpml",
    plate_model_dir+"/SA_Iran_Eurasia_Deformation Network.gpml",
    plate_model_dir+"/Sabzevar-Sistan_Topological Line.gpml",
    plate_model_dir+"/SCI_Subduction_points.gpml",
    plate_model_dir+"/SCI_Subduction_Topological_lines.gpml",
    plate_model_dir+"/Topological_lines.gpml",
    plate_model_dir+"/Topological_Points.gpml",
]

static_polygons = plate_model_dir+"/Global_EarthByte_GPlates_PresentDay_StaticPlatePolygons.shp"

plate_model = gplately.PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
    anchor_plate_id=anchor_plate_id,
)

coastlines = plate_model_dir+"/Global_coastlines_low_res.shp"
continents = plate_model_dir+"/Global_EarthByte_GPlates_PresentDay_ContinentalPolygons_edited 20.10.2025.gpml"
# continents = plate_model_dir+"/Global_EarthByte_GPlates_PresentDay_ContinentalPolygons.gpml"
COBs = plate_model_dir+"/Global_EarthByte_GeeK07_COB_Terranes_edited.gpml"

gplot = gplately.PlotTopologies(plate_model, continents=continents, COBs=COBs, coastlines=coastlines, anchor_plate_id=anchor_plate_id)

projection = ccrs.Mollweide(central_longitude=60)

In [ ]:
# # PMM
# pmm = PlateModelManager()
# pm = pmm.get_model(plate_model_name)

# rotation_model = pm.get_rotation_model()
# topology_features = pm.get_topologies()

# static_polygons = pm.get_static_polygons()
# coastlines = pm.get_coastlines()
# continents = pm.get_continental_polygons()
# COBs = pm.get_COBs()

# plate_model = gplately.PlateReconstruction(
#     rotation_model=rotation_model,
#     topology_features=topology_features,
#     static_polygons=static_polygons
# )

# gplot = gplately.PlotTopologies(plate_model, continents=continents, COBs=COBs, coastlines=coastlines)

# projection = ccrs.Mollweide(central_longitude=60)

In [ ]:
if not os.path.isdir(buffer_zones_dir):
    run_create_buffer_zones(
        times=time_steps,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        anchor_plate_id=anchor_plate_id,
        clip_to_overriding_plate=False,
        output_dir=buffer_zones_dir,
        buffer_distance=buffer_distance,
        n_jobs=n_jobs,
        verbose=True,
        return_output=False,
    )

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=projection, facecolor="azure")
    # ax.set_global()
    ax.set_extent([30, 90, 15, 45], crs=ccrs.PlateCarree())

    gplot.plot_continents(ax, edgecolor="none", facecolor="tan", alpha=0.5, zorder=1)
    gplot.plot_coastlines(ax, edgecolor="none", facecolor="tan", alpha=0.7, zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=5, spacingY=5, normalise=True, scale=50, alpha=0.1, zorder=3)

    buffer_zones_t_filename = f"buffer_zones_{time}Ma.geojson"
    buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    buffer_zones_t.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        facecolor="palegreen",
        edgecolor="none",
        alpha=0.7,
        zorder=4,
    )

    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)
    gplot.plot_trenches(ax, color="black", zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.015, color="black", zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)

    gl.top_labels=False
    # gl.bottom_labels=False
    gl.right_labels=False
    # gl.left_labels=False    
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}

    # ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    # ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    # ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)

    # Dummy handle to trigger custom handler
    trench_handle = Line2D([], [], color="black")
    
    # Add custom handles
    custom_handles = [
        Patch(facecolor="tan", edgecolor="none"),
        Patch(facecolor="palegreen", edgecolor="none", alpha=0.7),
        trench_handle,
        Line2D([0], [0], color="orangered", lw=2),
    ]
               
    custom_labels = [
        "Continental Crust",
        "Target Arc Environment",
        "Subduction zone",
        "Plate boundary",
    ]

    # Draw legend
    ax.legend(custom_handles, custom_labels, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.3),
              handler_map={trench_handle: HandlerTrenchLine()})

    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
    
    plt.show()

In [ ]:
if os.path.exists(buffer_zone_maps_dir):
    print(f"Buffer zone maps are located in {buffer_zone_maps_dir}")
else:
    os.makedirs(buffer_zone_maps_dir, exist_ok=True)
    
    generate_buffer_zone_maps(
        rotation_model,
        topology_features,
        static_polygons,
        coastlines,
        continents,
        COBs,
        projection,
        time_steps,
        buffer_zones_dir,
        output_dir=buffer_zone_maps_dir,
        anchor_plate_id=anchor_plate_id,
        n_jobs=n_jobs,
    )
    
    output_filenames = [
        os.path.join(buffer_zone_maps_dir, f"buffer_zone_map_{t:0.0f}Ma.png")
        for t in time_steps
    ]
    
    output_filename = os.path.join(outputs_dir, "buffer_zone_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )